In [ ]:

import os
import time
import pandas as pd
import torch
import gc
from google.colab import drive
import numpy as np

from torch.utils.data import DataLoader, TensorDataset, random_split
import copy

from data_loader import generate_topology
from main_cnn_GPU import run_simulation_CNN_GPU
from data_loader import set_seed
from data_loader import distribute_data
from data_loader import get_data
from theoretical_intensity import calculate_theoretical_intensity

# ==========================================
#
# ==========================================

#

#
NUM_CLIENTS = 20
BOOST_FACTORS = 2
#
train_ds, test_ds = get_data()
test_loader = DataLoader(test_ds, batch_size=256)

#


NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
ATK_TYPE = 'neurotoxin'
INTENSITY = 4
normfactor=0.5
seed_val=0
mal_ratio = 0.3
topo_type = 'scale_free'
mech = 'FedAvg'
set_seed(seed_val)

client_datasets = distribute_data(train_ds, NUM_CLIENTS)
G = generate_topology(NUM_CLIENTS, topo_type)
neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}
current_def_budget = 0
num_mal = int(NUM_CLIENTS*mal_ratio)
malicious_clients= list(np.random.choice(range(NUM_CLIENTS), int(NUM_CLIENTS*mal_ratio), replace=False))
defense_nodes = list()
theo_intensities = calculate_theoretical_intensity(
    neighbors, malicious_clients, NUM_CLIENTS, BOOST_FACTORS, lambda_benign=0.3
)
NORMFACTORS = [0.01,0.1,0.5,1,3]
SAVE_PATH = ''
for normfactor in NORMFACTORS:
    csv_filename = f"intensity_CNN_{topo_type}_MR{mal_ratio}_INTENSITY{INTENSITY}_{mech}_seed{seed_val}_normfactor2{normfactor}.csv"
    full_save_path = os.path.join(SAVE_PATH, csv_filename)
    log_filename = f"intensity_CNN_{topo_type}_MR{mal_ratio}_INTENSITY{INTENSITY}_{mech}_seed{seed_val}_normfactor2{normfactor}.txt"
    log_full_path = os.path.join(SAVE_PATH, log_filename)


    start_tick = time.time()

    _, _, accs, asrs = run_simulation_CNN_GPU(
        seed_val, NUM_CLIENTS, defense_nodes, malicious_clients,
        G, neighbors, client_datasets, test_loader,
        atk_type=ATK_TYPE,
        mechanism=mech,
        intensity=INTENSITY,
        norm_factor=normfactor,
        debug_mode=False,
        GLOBAL_ROUNDS=GLOBAL_ROUNDS,
        epochs=5, debug=False
    )
    end_tick = time.time()
    duration_sec = round(end_tick - start_tick, 2)
    #
    result_entry_base = {
        'seed': seed_val,
        'mechanism': mech,
        'malicious_ratio': mal_ratio,
        'topology': topo_type,
        'norm_factor': normfactor,
        'global_rounds': GLOBAL_ROUNDS,
        'duration_sec': duration_sec}
    all_results = []
    for i in range(NUM_CLIENTS):
        client_row = copy.deepcopy(result_entry_base)
        client_row.update({
            'client_id': i,
            'final_acc': accs[i],
            'final_asr': asrs[i],
            'theo_intensity': theo_intensities[i] if i < len(theo_intensities) else 0.0,
            'node_type': 'MAL' if i in malicious_clients else ('DEF' if i in defense_nodes else 'BEN'),
            'neighbors': str(list(neighbors.get(i, [])))
        })
        all_results.append(client_row)
    pd.DataFrame(all_results).to_csv(full_save_path, index=False)





In [ ]:
import os
import glob
import pandas as pd

from scipy.stats import kendalltau 

# ==========================================
# 
# ==========================================
print(f"\n{'='*40}")
print(f"{'='*40}")
SAVE_DIR = ''
#
fedavg_pattern = os.path.join(SAVE_DIR, "intensity_CNN*.csv")
fedavg_files = glob.glob(fedavg_pattern)
correlation_results = []
all_fedavg_nodes = []

for f in fedavg_files:
    df = pd.read_csv(f)

    # 1. 
    df_target = df.copy()

    # 2. 
    df_target['final_asr'] = pd.to_numeric(df_target['final_asr'], errors='coerce')
    df_target['theo_intensity'] = pd.to_numeric(df_target['theo_intensity'], errors='coerce')

    # 3. 
    df_target = df_target.dropna(subset=['final_asr', 'theo_intensity'])

    if len(df_target) > 1: 
        # 4.
        topo = df_target['topology'].iloc[0] if 'topology' in df_target.columns else "Unknown"
        mr = df_target['malicious_ratio'].iloc[0] if 'malicious_ratio' in df_target.columns else "Unknown"
        seed = df_target['seed'].iloc[0] if 'seed' in df_target.columns else "Unknown"

        # 5. 
        tau, p_value = kendalltau(df_target['final_asr'], df_target['theo_intensity'])

        # 6.
        correlation_results.append({
            'File': os.path.basename(f),
            'Topology': topo,
            'Mal_Ratio': mr,
            'Seed': seed,
            'Nodes_Count': len(df_target),
            'Kendall_Tau': tau,
            'P_Value': p_value,
            'Is_Significant': 'Yes' if p_value < 0.05 else 'No'
        })

        all_fedavg_nodes.append(df_target)



In [ ]:
# Kendall's tau
if correlation_results:
    df_results = pd.DataFrame(correlation_results)
    
    
    df_results = df_results.sort_values(by='Kendall_Tau', ascending=False)
    

    print(df_results[['Topology', 'Mal_Ratio', 'Seed', 'Kendall_Tau', 'P_Value', 'Is_Significant']].to_markdown(index=False))

   
    avg_tau = df_results['Kendall_Tau'].mean()
    significant_count = (df_results['P_Value'] < 0.05).sum()
    total_count = len(df_results)


In [ ]:
#Figure 3a
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# ==========================================
# 0. 
# ==========================================
TARGET_NORM_FACTOR = 0.5
TARGET_MECH = "FedAvg"
FOLDER_PATH = ''

file_pattern = os.path.join(FOLDER_PATH, f"*normfactor2{TARGET_NORM_FACTOR}.csv")

csv_files = glob.glob(file_pattern)

df_list = [pd.read_csv(f) for f in csv_files]
df_all = pd.concat(df_list, ignore_index=True)

# ==========================================
# 1. 
# ==========================================
if 'seed' not in df_all.columns:
    df_all['seed'] = np.repeat(np.arange(len(csv_files)), len(df_list[0]))

df_all['intensity_rank'] = df_all.groupby(['seed', 'node_type'])['theo_intensity'].rank(method='first')

df_avg = df_all.groupby(['node_type', 'intensity_rank']).agg({
    'theo_intensity': 'mean',
    'final_asr': 'mean',
    'final_acc': 'mean'
}).reset_index()

df_avg['final_asr'] = df_avg['final_asr']
df_avg['final_acc'] = df_avg['final_acc']

df_avg = df_avg.sort_values(by='theo_intensity').reset_index(drop=True)

df_avg['client_label'] = df_avg['node_type'] + '\n(Rank ' + df_avg['intensity_rank'].astype(int).astype(str) + ')'

colors_theo = ['#1F77B4' if nt != 'MAL' else '#002E5D' for nt in df_avg['node_type']]
colors_asr  = ['#FF7F0E' if nt != 'MAL' else '#D62728' for nt in df_avg['node_type']]

# ==========================================
# 2. 
# ==========================================
x = np.arange(len(df_avg))
width = 0.35

fig, ax1 = plt.subplots(figsize=(16, 7))

rects1 = ax1.bar(x - width/2, df_avg['theo_intensity'], width, color=colors_theo, edgecolor='white')
ax1.set_ylabel('Theoretical Diffusion Bound', color='#1F77B4', fontsize=14, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='#1F77B4', labelsize=12)
ax1.set_xlabel('Nodes Ranked by Theoretical Diffusion Bound', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(df_avg['client_label'], fontsize=11, rotation=45, ha='right')

for text in ax1.get_xticklabels():
    if 'MAL' in text.get_text():
        text.set_color('#D62728')
        text.set_fontweight('bold')

ax1.grid(True, axis='y', linestyle='--', alpha=0.4)

ax2 = ax1.twinx()
rects2 = ax2.bar(x + width/2, df_avg['final_asr'], width, color=colors_asr, edgecolor='white')

asr_max = df_avg['final_asr'].max()
is_percentage = asr_max > 1.5
ax2.set_ylabel(f'ASR {"(%)" if is_percentage else ""}', color='#D62728', fontsize=14, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#D62728', labelsize=12)
ax2.set_ylim(0, 105 if is_percentage else 1.05)

legend_elements = [
    Patch(facecolor='#1F77B4', edgecolor='w', label='Theoretical Diffusion Bound (BEN)'),
    Patch(facecolor='#002E5D', edgecolor='w', label='Theoretical Diffusion Bound (MAL)'),
    Patch(facecolor='#FF7F0E', edgecolor='w', label='ASR (BEN)'),
    Patch(facecolor='#D62728', edgecolor='w', label='ASR (MAL)')
]
ax1.legend(handles=legend_elements, loc='upper left', fontsize=11, frameon=True, shadow=True)

plt.title(fr'Comparison of Theoretical Diffusion Bound vs. ASR ($b_f$ = {TARGET_NORM_FACTOR})',
          fontsize=16, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

In [ ]:
#Figure 3b
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import snapshot_download
from google.colab import userdata

# ==========================================
#
# ==========================================
# 
df_list = [pd.read_csv(f) for f in csv_files]
df_all = pd.concat(df_list, ignore_index=True)

df_benign = df_all[df_all['node_type'] == 'BEN']
df_benign['final_asr'] = df_benign['final_asr']
df_benign['final_acc'] = df_benign['final_acc']
df_phase = df_benign.groupby('norm_factor')[['final_asr', 'final_acc']].mean().reset_index()

df_phase = df_phase.sort_values(by='norm_factor').reset_index(drop=True)

print("\n聚合后的相变数据：")
print(df_phase)

# ==========================================
# 
# ==========================================
fig, ax1 = plt.subplots(figsize=(10, 6))

#ax1.set_xscale('log')
ax1.set_xlabel(fr'$b_f$', fontsize=14, fontweight='bold')
ax1.set_xticks(df_phase['norm_factor'])
ax1.set_xticklabels(df_phase['norm_factor'], fontsize=9)

#
color_asr = '#D62728'
ax1.set_ylabel('Average ASR on Benign Nodes (%)', color=color_asr, fontsize=14, fontweight='bold')
line1 = ax1.plot(df_phase['norm_factor'], df_phase['final_asr'],
                 color=color_asr, marker='o', linewidth=3, markersize=8, label='ASR')
ax1.tick_params(axis='y', labelcolor=color_asr, labelsize=12)
ax1.set_ylim(-5, 105) 


ax1.grid(True, linestyle='--', alpha=0.6)

ax2 = ax1.twinx()
color_acc = '#1F77B4'
ax2.set_ylabel('Average ACC on Benign Nodes (%)', color=color_acc, fontsize=14, fontweight='bold')
line2 = ax2.plot(df_phase['norm_factor'], df_phase['final_acc'],
                 color=color_acc, marker='s', linestyle='--', linewidth=3, markersize=8, label='ACC')
ax2.tick_params(axis='y', labelcolor=color_acc, labelsize=12)
ax2.set_ylim(-5, 105)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center right', fontsize=12, frameon=True, shadow=True)

plt.title(fr'ACC, ASR vs. $b_f$', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()